# AgroVision enhanced experiment: EfficientNetV2B0 + ECA

This is a separate Colab notebook for the enhanced experiment. It leaves the MobileNetV2 baseline notebook, model, and report untouched. The primary model uses ImageNet EfficientNetV2B0, one lightweight Efficient Channel Attention (ECA) block, then a small classifier. A no-attention control uses the same backbone, preprocessing, augmentation, head, training split, and optimizer schedule.

The project’s PlantVillage config, experiment design, completed baseline training config, class labels, and seed-42 manifest are reused as inputs. The baseline training config is the source of truth for the completed baseline’s input size, batch size, seed, augmentation, and two-stage schedule.

EfficientNetV2 preprocessing matters: this notebook sets include_preprocessing=False and supplies the same external [-1, 1] scaling used by the baseline. Keras documents that EfficientNetV2 expects [-1, 1] input when its built-in preprocessing is disabled ([official documentation](https://keras.io/api/applications/efficientnet_v2/efficientnet_v2_models/)).

## Test-set policy

Manifest metadata is checked up front for integrity and partition counts, but test images are not decoded or used during training, ablation, early stopping, or checkpoint selection. Use validation-only ablation mode to compare the ECA and no-attention variants. Set final mode only after the configuration is selected; it evaluates the selected checkpoint on test once at the end. Do not use test metrics to choose between architectures or tune hyperparameters.

The local ZIP’s Kaggle origin/version and license were not verified in the Phase 2 audit. That limitation is carried into the machine-readable report.

## Drive layout

Keep the existing baseline artifacts in place. This notebook reads the baseline training config and class-label file; it reads baseline_report.json only in final mode and never writes to the baseline directory.

- MyDrive/AgroVision/dataset/plant_village_dataset.zip
- MyDrive/AgroVision/splits/plantvillage_seed42.csv
- MyDrive/AgroVision/config/plantvillage.json — copy from AgroVision_AI/ml/configs/
- MyDrive/AgroVision/config/experiments.json — copy from AgroVision_AI/ml/configs/
- MyDrive/AgroVision/models/baseline_mobilenetv2/training_config.json
- MyDrive/AgroVision/models/baseline_mobilenetv2/class_labels.json
- MyDrive/AgroVision/models/baseline_mobilenetv2/baseline_report.json — required only for final mode
- MyDrive/AgroVision/models/enhanced_efficientnetv2b0/ — new output root

For an attention ablation, run once with ECA enabled and once disabled in validation_ablation mode. Compare the resulting validation metrics only. Then set final mode for the selected configuration. Every run gets a timestamped output folder so prior enhanced runs are preserved.

In [ ]:
# Edit only these experiment switches. Defaults train the primary ECA model and reserve TEST for final evaluation.
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/AgroVision")
DATASET_ZIP = DRIVE_ROOT / "dataset/plant_village_dataset.zip"
SPLIT_CSV = DRIVE_ROOT / "splits/plantvillage_seed42.csv"
PROJECT_CONFIG_JSON = DRIVE_ROOT / "config/plantvillage.json"
EXPERIMENTS_CONFIG_JSON = DRIVE_ROOT / "config/experiments.json"
BASELINE_DIR = DRIVE_ROOT / "models/baseline_mobilenetv2"
BASELINE_TRAINING_CONFIG = BASELINE_DIR / "training_config.json"
BASELINE_CLASS_LABELS = BASELINE_DIR / "class_labels.json"
BASELINE_REPORT_JSON = BASELINE_DIR / "baseline_report.json"
ENHANCED_OUTPUT_ROOT = DRIVE_ROOT / "models/enhanced_efficientnetv2b0"
EXTRACT_DIR = Path("/content/agrovision_plantvillage_enhanced/extracted")

USE_ECA_ATTENTION = True  # True = primary ECA model; False = no-attention control
RUN_MODE = "final"        # "validation_ablation" never reads test images; "final" evaluates test once
FINE_TUNE_FRACTION_OVERRIDE = None  # None reuses the baseline training config (upper 30%)
''

In [ ]:
import hashlib, json, os, platform, random, shutil, stat, time, traceback, zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import drive
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

if RUN_MODE not in {"final", "validation_ablation"}:
    raise ValueError("RUN_MODE must be 'final' or 'validation_ablation'.")
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError(
        "No TensorFlow GPU was detected. In Colab select Runtime > Change runtime type > GPU, "
        "reconnect, and run again. This notebook is designed for a Colab T4-class GPU."
    )
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
    print("GPU:", tf.config.experimental.get_device_details(gpu).get("device_name", gpu.name))

# Match baseline seeding and request deterministic TensorFlow operations where available.
random.seed(42)
np.random.seed(42)
tf.keras.utils.set_random_seed(42)
determinism_enabled = True
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError) as exc:
    determinism_enabled = False
    print("Could not enable deterministic ops:", str(exc))

print("Python:", platform.python_version())
drive.mount("/content/drive")

required_paths = [DATASET_ZIP, SPLIT_CSV, PROJECT_CONFIG_JSON, EXPERIMENTS_CONFIG_JSON,
                  BASELINE_TRAINING_CONFIG, BASELINE_CLASS_LABELS]
if RUN_MODE == "final":
    required_paths.append(BASELINE_REPORT_JSON)
for path in required_paths:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required Drive file: {path}. See the Drive layout above.")

with open(PROJECT_CONFIG_JSON, encoding="utf-8") as f:
    project_config = json.load(f)
with open(EXPERIMENTS_CONFIG_JSON, encoding="utf-8") as f:
    experiments_config = json.load(f)
with open(BASELINE_TRAINING_CONFIG, encoding="utf-8") as f:
    baseline_config = json.load(f)
with open(BASELINE_CLASS_LABELS, encoding="utf-8") as f:
    labels_by_id = {int(k): str(v) for k, v in json.load(f).items()}

# Reuse actual baseline protocol; config files are checked against each other.
shared = experiments_config["shared_protocol"]
IMG_SIZE = int(baseline_config["IMG_SIZE"])
BATCH_SIZE = int(baseline_config["BATCH_SIZE"])
SEED = int(baseline_config["seed"])
HEAD_EPOCHS = int(baseline_config["HEAD_EPOCHS"])
FINETUNE_EPOCHS = int(baseline_config["FINETUNE_EPOCHS"])
HEAD_LEARNING_RATE = float(baseline_config["HEAD_LEARNING_RATE"])
FINETUNE_LEARNING_RATE = float(baseline_config["FINETUNE_LEARNING_RATE"])
DROPOUT = float(baseline_config["DROPOUT"])
PATIENCE = int(baseline_config["PATIENCE"])
FINE_TUNE_FRACTION = float(FINE_TUNE_FRACTION_OVERRIDE or baseline_config["FINE_TUNE_FRACTION"])
AUGMENTATION_CONFIG = baseline_config["augmentation"]
NUM_CLASSES = int(project_config["labels"]["class_count"])
EXPECTED_IMAGE_COUNT = int(project_config["archive"]["image_count"])
EXPECTED_ARCHIVE_SHA256 = project_config["archive"]["sha256"]
variant_name = "EfficientNetV2B0_ECA" if USE_ECA_ATTENTION else "EfficientNetV2B0_control"
variant_folder = "eca" if USE_ECA_ATTENTION else "no_attention"
candidate_names = {item["name"] for item in experiments_config["enhanced_candidates"]}
if variant_name not in candidate_names:
    raise ValueError(f"{variant_name} is not defined in experiments.json.")
if SEED != int(project_config["seed"]) or SEED != int(shared["seed"]):
    raise ValueError("Baseline, dataset, and experiment config seeds differ.")
if [IMG_SIZE, IMG_SIZE] != project_config["image_contract"]["target_dimensions"]:
    raise ValueError("Baseline image size differs from the audited project image contract.")
if [IMG_SIZE, IMG_SIZE] != shared["input_size"] or NUM_CLASSES != len(labels_by_id):
    raise ValueError("Project, experiment, or class-label configuration mismatch.")
if experiments_config["status"].startswith("design-only") is False:
    print("Experiment config status:", experiments_config["status"])

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_OUTPUT_DIR = ENHANCED_OUTPUT_ROOT / variant_folder / timestamp
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
AUTOTUNE = tf.data.AUTOTUNE
print("Variant:", variant_name, "| mode:", RUN_MODE, "| output:", RUN_OUTPUT_DIR)
print("Reused baseline protocol:", {k: baseline_config[k] for k in
      ["IMG_SIZE", "BATCH_SIZE", "HEAD_EPOCHS", "FINETUNE_EPOCHS", "HEAD_LEARNING_RATE", "FINETUNE_LEARNING_RATE", "DROPOUT", "seed"]})

## Verify and extract the audited dataset

The notebook checks the ZIP’s audited SHA-256, actual image inventory, train/validation manifest paths, all 38 class folders, class IDs/labels, the exact baseline split counts, and duplicate-group isolation. It extracts into a dedicated temporary Colab directory and does not modify the Drive ZIP or baseline artifacts.

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_extract_zip(zip_path, destination):
    """Reject ZIP traversal/absolute paths and symlinks while extracting."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    suffixes = {".jpg", ".jpeg", ".png", ".bmp"}
    images = 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            member = PurePosixPath(info.filename.replace("\\", "/"))
            if member.is_absolute() or not member.parts or any(part in {"..", ""} for part in member.parts):
                raise ValueError(f"Unsafe ZIP path: {info.filename!r}")
            target = (destination / Path(*member.parts)).resolve()
            try:
                target.relative_to(root)
            except ValueError as exc:
                raise ValueError(f"ZIP entry escapes extraction directory: {info.filename!r}") from exc
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if ((info.external_attr >> 16) & 0o170000) == stat.S_IFLNK:
                raise ValueError(f"ZIP symlink is not allowed: {info.filename!r}")
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info, "r") as source, open(target, "wb") as output:
                shutil.copyfileobj(source, output, length=1024 * 1024)
            if Path(member.name).suffix.lower() in suffixes:
                images += 1
    return images


archive_sha256 = sha256_file(DATASET_ZIP)
print("ZIP SHA-256:", archive_sha256)
if archive_sha256.lower() != EXPECTED_ARCHIVE_SHA256.lower():
    raise ValueError("ZIP SHA-256 differs from ml/configs/plantvillage.json; verify the archive copy.")
manifest_sha256 = sha256_file(SPLIT_CSV)
baseline_manifest_sha256 = baseline_config.get("split_manifest_sha256")
if baseline_manifest_sha256 and manifest_sha256.lower() != baseline_manifest_sha256.lower():
    raise ValueError("Drive split CSV differs from the split used by the completed baseline.")
archive_image_count = safe_extract_zip(DATASET_ZIP, EXTRACT_DIR)
if archive_image_count != EXPECTED_IMAGE_COUNT:
    raise ValueError(f"ZIP contains {archive_image_count} images; project config expects {EXPECTED_IMAGE_COUNT}.")
print("Verified ZIP image inventory:", archive_image_count)

In [ ]:
manifest = pd.read_csv(SPLIT_CSV, dtype={"archive_member_path": str, "archive_class_name": str, "class_label": str})
required_columns = {"archive_member_path", "archive_class_name", "class_id", "class_label", "split",
                    "duplicate_group_id", "duplicate_group_size", "duplicate_group_reason"}
if required_columns - set(manifest.columns):
    raise ValueError(f"Manifest is missing columns: {sorted(required_columns - set(manifest.columns))}")
manifest["class_id"] = pd.to_numeric(manifest["class_id"], errors="raise").astype(int)
allowed_splits = {"train", "validation", "test"}
if set(manifest["split"]) != allowed_splits or manifest["archive_member_path"].duplicated().any():
    raise ValueError("Manifest split names or image paths are invalid.")

expected_counts = {k: int(v) for k, v in baseline_config["partition_counts"].items()}
partition_counts = manifest["split"].value_counts().to_dict()
if len(manifest) != EXPECTED_IMAGE_COUNT or partition_counts != expected_counts:
    raise ValueError(f"Split counts differ from baseline/config: got {partition_counts}, expected {expected_counts}.")
class_ids = sorted(labels_by_id)
if class_ids != list(range(NUM_CLASSES)):
    raise ValueError(f"Expected class IDs 0..{NUM_CLASSES - 1}; found {class_ids}")

folder_to_id = {str(k): int(v) for k, v in project_config["labels"]["source_folder_to_class_id"].items()}
folder_aliases = project_config["labels"]["source_folder_to_label_aliases"]
if len(folder_to_id) != NUM_CLASSES:
    raise ValueError("Project source-folder map does not contain exactly 38 folders.")
for row in manifest[["archive_class_name", "class_id", "class_label"]].drop_duplicates().itertuples(index=False):
    folder = str(row.archive_class_name)
    expected_id = folder_to_id.get(folder)
    expected_label = folder_aliases.get(folder, folder)
    if expected_id != int(row.class_id) or expected_label != str(row.class_label):
        raise ValueError(f"Manifest folder/label mapping mismatch for {folder!r}.")
    if labels_by_id[int(row.class_id)] != str(row.class_label):
        raise ValueError(f"class_labels.json mismatch for class ID {row.class_id}.")

archive_paths = manifest["archive_member_path"].map(PurePosixPath)
if any(p.is_absolute() or len(p.parts) != 2 or ".." in p.parts for p in archive_paths):
    raise ValueError("Manifest contains an unsafe or unexpected archive-relative path.")
if any(p.parts[0] != row.archive_class_name for p, row in zip(archive_paths, manifest.itertuples(index=False))):
    raise ValueError("Manifest class folder differs from its archive path.")
manifest["image_path"] = [str((EXTRACT_DIR / Path(*p.parts)).resolve()) for p in archive_paths]
train_validation_paths = manifest.loc[manifest.split != "test", "image_path"]
missing = [p for p in train_validation_paths if not Path(p).is_file()]
if missing:
    raise FileNotFoundError(f"{len(missing)} train/validation image paths are missing; example: {missing[0]}")

suffixes = {".jpg", ".jpeg", ".png", ".bmp"}
extracted_images = [p for p in EXTRACT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in suffixes]
folders = {p.relative_to(EXTRACT_DIR).parts[0] for p in extracted_images}
if len(extracted_images) != EXPECTED_IMAGE_COUNT or folders != set(folder_to_id):
    raise ValueError("Extracted image inventory or class folders differ from project config.")
if (manifest.groupby("duplicate_group_id")["split"].nunique() > 1).any():
    raise ValueError("At least one duplicate group crosses partitions.")
if any(manifest.loc[manifest.split == part, "class_id"].nunique() != NUM_CLASSES for part in allowed_splits):
    raise ValueError("Every split must contain all 38 classes.")

train_df = manifest.loc[manifest.split == "train"].reset_index(drop=True)
validation_df = manifest.loc[manifest.split == "validation"].reset_index(drop=True)
test_count_verified = int((manifest.split == "test").sum())  # metadata only; no test image paths are opened
class_distribution_by_split = (manifest.groupby(["split", "class_id", "class_label"]).size()
                               .rename("image_count").reset_index())
print("Images:", len(extracted_images), "| folders/classes:", len(folders), "| split counts:", partition_counts)
print("Manifest labels, project folder aliases, class-label IDs, and duplicate-group isolation verified.")

## Same input and augmentation protocol; EfficientNetV2-specific preprocessing

The data pipeline emits RGB float pixels in [0, 255]. The model reproduces the completed baseline's train-only augmentation, then scales pixels to [-1, 1]. EfficientNetV2B0 is created with include_preprocessing=False, so it receives this shared input range without double-normalization. Validation and test images are resized only; no augmentation or shuffle is applied to them.

In [ ]:
# Train-only model-side augmentation keeps evaluation deterministic and leaves the input pipeline shared.
from tensorflow.keras import layers

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(float(AUGMENTATION_CONFIG["rotation"]), fill_mode="reflect", seed=SEED + 1),
    layers.RandomTranslation(float(AUGMENTATION_CONFIG["translation"]),
                             float(AUGMENTATION_CONFIG["translation"]), fill_mode="reflect", seed=SEED + 2),
    layers.RandomZoom(float(AUGMENTATION_CONFIG["zoom"]), fill_mode="reflect", seed=SEED + 3),
    layers.RandomContrast(float(AUGMENTATION_CONFIG["contrast"]), seed=SEED + 4),
], name="train_only_augmentation")


def make_dataset(frame, training=False):
    paths = frame["image_path"].astype(str).to_numpy()
    class_ids_array = frame["class_id"].to_numpy(dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, class_ids_array))
    if training:
        ds = ds.shuffle(min(len(frame), 10000), seed=SEED, reshuffle_each_iteration=True)

    def decode_resize(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], method="bilinear")
        return tf.cast(image, tf.float32), label  # 0..255; model performs shared baseline scaling

    ds = ds.map(decode_resize, num_parallel_calls=AUTOTUNE, deterministic=True)
    return ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
validation_ds = make_dataset(validation_df)
train_counts = train_df["class_id"].value_counts().reindex(class_ids, fill_value=0).sort_index()
weights = compute_class_weight("balanced", classes=np.asarray(class_ids), y=train_df["class_id"].to_numpy())
class_weights = {int(i): float(w) for i, w in zip(class_ids, weights)}
print("Train-only class distribution:")
print(pd.DataFrame({"class_id": class_ids, "label": [labels_by_id[i] for i in class_ids],
                    "train_images": train_counts.values}).to_string(index=False))
print("Balanced class weights:", json.dumps(class_weights, indent=2))

## EfficientNetV2B0 with optional ECA

ECA is placed on the final spatial feature tensor, immediately after EfficientNetV2B0 and before global pooling and the classifier head. It globally averages each channel, applies a tiny 1D convolution across neighboring channel descriptors, and uses sigmoid weights to rescale the original channels. This adds only a small channel-interaction layer and avoids extra residual, spatial-attention, or convolution blocks. Set ECA off for the control; all other layers and training choices remain the same.

In [ ]:
@tf.keras.utils.register_keras_serializable(package="AgroVision")
class ECAChannelAttention(tf.keras.layers.Layer):
    """Lightweight ECA: local channel interaction followed by sigmoid channel reweighting."""
    def __init__(self, kernel_size=3, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = int(kernel_size)
        self.channel_conv = tf.keras.layers.Conv1D(
            filters=1, kernel_size=self.kernel_size, padding="same",
            use_bias=False, name="local_channel_interaction"
        )

    def call(self, inputs):
        # (batch, height, width, channels) -> pooled (batch, channels)
        descriptor = tf.reduce_mean(inputs, axis=[1, 2])
        # Conv1D treats channels as sequence positions; its tiny kernel learns neighboring channel interactions.
        gates = tf.sigmoid(tf.squeeze(self.channel_conv(tf.expand_dims(descriptor, axis=-1)), axis=-1))
        gates = gates[:, None, None, :]
        return inputs * gates

    def get_config(self):
        config = super().get_config()
        config.update({"kernel_size": self.kernel_size})
        return config


def build_model(use_attention):
    raw = layers.Input((IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32, name="rgb_pixels_0_255")
    x = layers.Rescaling(1.0 / 255.0, name="pixels_0_to_1")(raw)
    x = augmentation(x)
    x = layers.Rescaling(2.0, offset=-1.0, name="shared_baseline_preprocessing_minus1_to_1")(x)
    # Keep the application default name so Keras selects its registered B0 block configuration.
    backbone = tf.keras.applications.EfficientNetV2B0(
        include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_preprocessing=False
    )
    backbone.trainable = False
    features = backbone(x, training=False)
    # ECA sits before the classification head. Control path deliberately omits only this layer.
    if use_attention:
        features = ECAChannelAttention(kernel_size=3, name="eca_channel_attention")(features)
    y = layers.GlobalAveragePooling2D(name="global_average_pooling")(features)
    y = layers.BatchNormalization(name="head_batch_normalization")(y)
    y = layers.Dense(256, activation="relu", name="head_dense_256")(y)
    y = layers.Dropout(DROPOUT, seed=SEED, name="head_dropout")(y)
    logits = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="class_probabilities")(y)
    return tf.keras.Model(raw, logits, name=f"agrovision_{variant_folder}")


def parameter_counts(model):
    total = int(sum(np.prod(weight.shape) for weight in model.weights))
    trainable = int(sum(np.prod(weight.shape) for weight in model.trainable_weights))
    return {"total": total, "trainable": trainable, "non_trainable": total - trainable}


def write_model_summary(model, path):
    lines = []
    model.summary(print_fn=lines.append)
    counts = parameter_counts(model)
    lines += ["", f"Input shape: {model.input_shape}", f"Output shape: {model.output_shape}",
              f"Total parameters: {counts['total']:,}", f"Trainable parameters: {counts['trainable']:,}",
              f"Non-trainable parameters: {counts['non_trainable']:,}"]
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("\n".join(lines[-6:]))
    return counts

model = build_model(USE_ECA_ATTENTION)
model.compile(optimizer=tf.keras.optimizers.Adam(HEAD_LEARNING_RATE),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
stage1_initial_counts = write_model_summary(model, RUN_OUTPUT_DIR / "model_summary.txt")
print("Input range is raw RGB 0..255; model-side output before backbone is [-1, 1].")

## Two-stage training

The schedule is loaded from the completed baseline training config for a comparable transfer-learning protocol. Stage 1 freezes the EfficientNetV2B0 backbone. Stage 2 restores the best Stage 1 validation checkpoint, unfreezes its upper configured fraction, keeps BatchNormalization frozen, and recompiles with the smaller fine-tuning learning rate. Only train and validation datasets are passed to fit or callbacks.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

best_model_path = RUN_OUTPUT_DIR / "best_model.keras"
stage1_path = RUN_OUTPUT_DIR / "stage1_best.keras"

def callbacks_for(path):
    return [
        ModelCheckpoint(str(path), monitor="val_loss", mode="min", save_best_only=True, verbose=1),
        EarlyStopping(monitor="val_loss", mode="min", patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.3,
                          patience=2, min_lr=1e-7, verbose=1),
    ]

run_started_utc = datetime.now(timezone.utc).isoformat()
training_config = {
    "run_started_utc": run_started_utc, "variant": variant_name, "use_eca_attention": USE_ECA_ATTENTION,
    "run_mode": RUN_MODE, "seed": SEED, "dataset_config_path": str(PROJECT_CONFIG_JSON),
    "experiments_config_path": str(EXPERIMENTS_CONFIG_JSON), "dataset_zip_path": str(DATASET_ZIP),
    "dataset_zip_sha256": archive_sha256, "split_manifest_path": str(SPLIT_CSV),
    "split_manifest_sha256": manifest_sha256, "class_labels_path": str(BASELINE_CLASS_LABELS),
    "verified_image_count": len(manifest), "verified_class_count": NUM_CLASSES,
    "partition_counts": {k: int(v) for k, v in partition_counts.items()},
    "train_class_distribution": class_distribution_by_split.loc[class_distribution_by_split.split == "train"].to_dict(orient="records"),
    "IMG_SIZE": IMG_SIZE, "BATCH_SIZE": BATCH_SIZE, "HEAD_EPOCHS": HEAD_EPOCHS,
    "FINETUNE_EPOCHS": FINETUNE_EPOCHS, "HEAD_LEARNING_RATE": HEAD_LEARNING_RATE,
    "FINETUNE_LEARNING_RATE": FINETUNE_LEARNING_RATE, "DROPOUT": DROPOUT,
    "FINE_TUNE_FRACTION": FINE_TUNE_FRACTION, "PATIENCE": PATIENCE,
    "augmentation": AUGMENTATION_CONFIG,
    "preprocessing": "RGB float32 0..255; divide by 255, train-only augmentation, then x*2-1; EfficientNetV2 include_preprocessing=False",
    "class_weight_strategy": "sklearn compute_class_weight(class_weight='balanced') on train split only",
    "class_weights": None, "parameter_counts_stage1": stage1_initial_counts,
    "tensorflow_version": tf.__version__, "python_version": platform.python_version(),
    "gpu_names": [tf.config.experimental.get_device_details(g).get("device_name", g.name) for g in gpus],
    "deterministic_ops_enabled": determinism_enabled,
}
# Save a reproducible config before the potentially long fit.
(RUN_OUTPUT_DIR / "training_config.json").write_text(json.dumps(training_config, indent=2), encoding="utf-8")
shutil.copy2(PROJECT_CONFIG_JSON, RUN_OUTPUT_DIR / "plantvillage_config.json")
shutil.copy2(EXPERIMENTS_CONFIG_JSON, RUN_OUTPUT_DIR / "experiments_config.json")
shutil.copy2(BASELINE_CLASS_LABELS, RUN_OUTPUT_DIR / "class_labels.json")
shutil.copy2(SPLIT_CSV, RUN_OUTPUT_DIR / "split_manifest.csv")
class_distribution_by_split.to_csv(RUN_OUTPUT_DIR / "class_distribution_by_split.csv", index=False)

train_started = time.perf_counter()
stage_histories = []
try:
    print("STAGE 1 — frozen EfficientNetV2B0 backbone")
    started = time.perf_counter()
    history1 = model.fit(train_ds, validation_data=validation_ds, epochs=HEAD_EPOCHS,
                         class_weight=class_weights, callbacks=callbacks_for(stage1_path), verbose=1)
    stage_histories.append({"stage": "frozen_backbone", "elapsed_seconds": time.perf_counter() - started,
                            "history": {k: [float(v) for v in values] for k, values in history1.history.items()}})

    model = tf.keras.models.load_model(stage1_path)
    backbone = model.get_layer("efficientnetv2-b0")  # EfficientNetV2B0 application default name
    backbone.trainable = True
    first_trainable = int(len(backbone.layers) * (1.0 - FINE_TUNE_FRACTION))
    for i, layer in enumerate(backbone.layers):
        layer.trainable = i >= first_trainable and not isinstance(layer, tf.keras.layers.BatchNormalization)
    trainable_backbone_layers = sum(bool(layer.trainable) for layer in backbone.layers)
    if trainable_backbone_layers == 0:
        raise RuntimeError("Fine-tuning selected no trainable EfficientNetV2B0 layers.")
    print(f"Fine-tuning upper {FINE_TUNE_FRACTION:.0%}: {trainable_backbone_layers}/{len(backbone.layers)} layers; BatchNormalization frozen.")
    model.compile(optimizer=tf.keras.optimizers.Adam(FINETUNE_LEARNING_RATE),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    print("STAGE 2 — controlled fine-tuning")
    started = time.perf_counter()
    history2 = model.fit(train_ds, validation_data=validation_ds, epochs=FINETUNE_EPOCHS,
                         class_weight=class_weights, callbacks=callbacks_for(best_model_path), verbose=1)
    stage_histories.append({"stage": "fine_tuning", "elapsed_seconds": time.perf_counter() - started,
                            "history": {k: [float(v) for v in values] for k, values in history2.history.items()}})
    training_seconds = time.perf_counter() - train_started

    # Restore and freeze selection at the best validation checkpoint. No further fit calls occur.
    model = tf.keras.models.load_model(best_model_path)
    final_parameter_counts = write_model_summary(model, RUN_OUTPUT_DIR / "model_summary.txt")
    model.save(best_model_path)
    h5_model_path = RUN_OUTPUT_DIR / "best_model.h5"
    h5_saved = False
    try:
        model.save(h5_model_path)
        h5_saved = True
        print("Optional HDF5 checkpoint saved.")
    except Exception as exc:
        print("HDF5 export unavailable in this runtime:", repr(exc))
        if h5_model_path.exists():
            h5_model_path.unlink()
except Exception:
    (RUN_OUTPUT_DIR / "training_error.txt").write_text(traceback.format_exc(), encoding="utf-8")
    raise

training_history = {"run_started_utc": run_started_utc, "training_elapsed_seconds": float(training_seconds),
                    "stages": stage_histories}
(RUN_OUTPUT_DIR / "training_history.json").write_text(json.dumps(training_history, indent=2), encoding="utf-8")
training_config.update({
    "class_weights": {str(k): v for k, v in class_weights.items()},
    "parameter_counts_after_fine_tuning": final_parameter_counts,
    "training_elapsed_seconds": float(training_seconds), "h5_export_saved": h5_saved,
})
(RUN_OUTPUT_DIR / "training_config.json").write_text(json.dumps(training_config, indent=2), encoding="utf-8")
print(f"Actual training time: {training_seconds / 60:.2f} minutes")

In [ ]:
preprocessing_config = {
    "input_shape": [IMG_SIZE, IMG_SIZE, 3], "color_order": "RGB", "input_dtype": "float32",
    "raw_input_range": [0.0, 255.0], "augmentation": AUGMENTATION_CONFIG,
    "augmentation_training_only": True,
    "external_normalization": "(x / 255) * 2 - 1, equivalent to x / 127.5 - 1",
    "efficientnetv2_include_preprocessing": False,
    "validation_test_augmentation": False,
}
(RUN_OUTPUT_DIR / "preprocessing_config.json").write_text(json.dumps(preprocessing_config, indent=2), encoding="utf-8")

# Plot real histories from the two training stages.
curves = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": []}
for stage in stage_histories:
    for key in curves:
        curves[key].extend(stage["history"].get(key, []))
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(curves["accuracy"], label="train")
axes[0].plot(curves["val_accuracy"], label="validation")
axes[0].set(title="Accuracy by epoch", xlabel="Epoch (Stage 1 then Stage 2)", ylabel="Accuracy")
axes[0].legend()
axes[1].plot(curves["loss"], label="train")
axes[1].plot(curves["val_loss"], label="validation")
axes[1].set(title="Loss by epoch", xlabel="Epoch (Stage 1 then Stage 2)", ylabel="Loss")
axes[1].legend()
fig.tight_layout()
fig.savefig(RUN_OUTPUT_DIR / "training_validation_curves.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## Validation metrics and ablation comparison

The best validation-loss checkpoint is now fixed. Validation results include accuracy, macro and weighted precision/recall/F1, top-3 accuracy, per-class scores, and confusion matrix. Validation-only ablation runs stop without creating or reading a test dataset.

In [ ]:
best_model = tf.keras.models.load_model(best_model_path, compile=False)
class_names = [labels_by_id[i] for i in class_ids]

def evaluate_and_save(model_to_eval, dataset, true_ids, split_name):
    probs = model_to_eval.predict(dataset, verbose=1)
    predictions = np.argmax(probs, axis=1)
    top3 = np.argsort(probs, axis=1)[:, -3:]
    report = classification_report(true_ids, predictions, labels=class_ids,
                                   target_names=class_names, output_dict=True, zero_division=0)
    matrix = confusion_matrix(true_ids, predictions, labels=class_ids)
    per_class = [{
        "class_id": int(i), "class_label": name,
        "precision": float(report[name]["precision"]), "recall": float(report[name]["recall"]),
        "f1_score": float(report[name]["f1-score"]), "support": int(report[name]["support"]),
    } for i, name in zip(class_ids, class_names)]
    metrics = {
        "result_split": split_name, "sample_count": int(len(true_ids)),
        "accuracy": float(accuracy_score(true_ids, predictions)),
        "precision_macro": float(report["macro avg"]["precision"]),
        "recall_macro": float(report["macro avg"]["recall"]),
        "f1_macro": float(report["macro avg"]["f1-score"]),
        "precision_weighted": float(report["weighted avg"]["precision"]),
        "recall_weighted": float(report["weighted avg"]["recall"]),
        "f1_weighted": float(report["weighted avg"]["f1-score"]),
        "top3_accuracy": float(np.mean([int(label) in row for label, row in zip(true_ids, top3)])),
        "per_class_metrics": per_class, "confusion_matrix": matrix.tolist(),
        "class_label_order": class_names,
    }
    stem = split_name.lower()
    (RUN_OUTPUT_DIR / f"{stem}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    pd.DataFrame(per_class).to_csv(RUN_OUTPUT_DIR / f"{stem}_per_class_metrics.csv", index=False)
    pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv(
        RUN_OUTPUT_DIR / f"{stem}_confusion_matrix.csv", index_label="actual_class")
    print(split_name, json.dumps({k: v for k, v in metrics.items()
          if k not in {"per_class_metrics", "confusion_matrix", "class_label_order"}}, indent=2))
    return metrics

validation_ids = validation_df["class_id"].to_numpy(dtype=np.int32)
validation_metrics = evaluate_and_save(best_model, validation_ds, validation_ids, "VALIDATION")

# Compare validation-only artifacts from previous ECA/control ablation runs, if available.
ablation_rows = []
for path in ENHANCED_OUTPUT_ROOT.rglob("validation_metrics.json"):
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        run_config = json.loads((path.parent / "training_config.json").read_text(encoding="utf-8"))
        ablation_rows.append({
            "run_dir": str(path.parent), "variant": run_config.get("variant"),
            "run_mode": run_config.get("run_mode"),
            **{key: data[key] for key in ["accuracy", "precision_macro", "recall_macro",
                                           "f1_macro", "f1_weighted", "top3_accuracy"]},
        })
    except (OSError, KeyError, json.JSONDecodeError):
        continue
ablation_comparison = {"comparison_basis": "validation only; no test metrics read",
                       "runs": sorted(ablation_rows, key=lambda item: item["run_dir"])}
(RUN_OUTPUT_DIR / "validation_ablation_comparison.json").write_text(
    json.dumps(ablation_comparison, indent=2), encoding="utf-8")
print("Saved validation-only ECA/control comparison:", len(ablation_rows), "run(s)")

## Efficiency measurements

Parameter counts, model file size, batch-one GPU and CPU latency, and average latency over repeated individual images are measured using validation images only. This does not read the held-out test partition.

In [ ]:
def measure_batch1(model_to_time, device, image_batch, iterations=100, warmup=10):
    samples = []
    with tf.device(device):
        one = tf.identity(image_batch[:1])
        for _ in range(warmup):
            model_to_time(one, training=False).numpy()
        for _ in range(iterations):
            start = time.perf_counter()
            model_to_time(one, training=False).numpy()  # synchronize device before stopping timer
            samples.append((time.perf_counter() - start) * 1000.0)
    return {"device": device, "batch_size": 1, "iterations": iterations,
            "mean_ms_per_image": float(np.mean(samples)),
            "median_ms_per_image": float(np.median(samples)),
            "p95_ms_per_image": float(np.percentile(samples, 95))}

parameter_summary = parameter_counts(model)
sample_batch, _ = next(iter(validation_ds))
gpu_latency = measure_batch1(best_model, "/GPU:0", sample_batch)
with tf.device("/CPU:0"):
    cpu_model = tf.keras.models.load_model(best_model_path, compile=False)
cpu_latency = measure_batch1(cpu_model, "/CPU:0", sample_batch)
efficiency = {
    "total_parameters": parameter_summary["total"],
    "trainable_parameters_after_fine_tuning": parameter_summary["trainable"],
    "non_trainable_parameters_after_fine_tuning": parameter_summary["non_trainable"],
    "best_model_keras_size_bytes": int(best_model_path.stat().st_size),
    "best_model_keras_size_mib": float(best_model_path.stat().st_size / (1024 ** 2)),
    "gpu_name": tf.config.experimental.get_device_details(gpus[0]).get("device_name", gpus[0].name),
    "gpu_batch1_latency": gpu_latency, "cpu_batch1_latency": cpu_latency,
}
if h5_saved and h5_model_path.exists():
    efficiency["best_model_h5_size_bytes"] = int(h5_model_path.stat().st_size)
(RUN_OUTPUT_DIR / "efficiency_metrics.json").write_text(json.dumps(efficiency, indent=2), encoding="utf-8")
print(json.dumps(efficiency, indent=2))

## Final TEST evaluation — final mode only

The only test-image evaluation is below and runs only when RUN_MODE is final. Use validation_ablation mode for control experiments; that branch cannot create a test dataset. Final mode evaluates the already selected checkpoint once and writes enhanced metrics plus a read-only comparison with the baseline report. No test metrics are used by training, stopping, or checkpoint selection.

In [ ]:
test_metrics = None
baseline_test_metrics = None
if RUN_MODE == "final":
    # First test-image access: after training, checkpoint selection, and validation/efficiency analysis.
    test_df = manifest.loc[manifest.split == "test"].reset_index(drop=True)
    if len(test_df) != test_count_verified or len(test_df) != partition_counts["test"]:
        raise RuntimeError("Verified test partition count changed before final evaluation.")
    missing_test = [p for p in test_df["image_path"] if not Path(p).is_file()]
    if missing_test:
        raise FileNotFoundError(f"{len(missing_test)} test image paths are missing; example: {missing_test[0]}")
    test_ds = make_dataset(test_df, training=False)
    test_ids = test_df["class_id"].to_numpy(dtype=np.int32)
    test_metrics = evaluate_and_save(best_model, test_ds, test_ids, "TEST")

    # Read the baseline report only after enhanced model selection and final test evaluation.
    with open(BASELINE_REPORT_JSON, encoding="utf-8") as stream:
        baseline_report = json.load(stream)
    baseline_test_metrics = baseline_report.get("test_results")
    if not baseline_test_metrics or int(baseline_test_metrics.get("sample_count", -1)) != len(test_ids):
        raise ValueError("Baseline report has no compatible held-out test metrics.")
    comparison = {
        "comparison_policy": "same seed-42 test partition; baseline values read from the existing report; no baseline file is modified",
        "baseline_report_path": str(BASELINE_REPORT_JSON),
        "baseline_test_results": baseline_test_metrics,
        "enhanced_variant": variant_name,
        "enhanced_test_results": test_metrics,
        "absolute_metric_differences_enhanced_minus_baseline": {
            key: float(test_metrics[key] - baseline_test_metrics[key])
            for key in ["accuracy", "precision_macro", "recall_macro", "f1_macro",
                        "precision_weighted", "recall_weighted", "f1_weighted", "top3_accuracy"]
        },
    }
    (RUN_OUTPUT_DIR / "baseline_vs_enhanced.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")

enhanced_report = {
    "report_generated_utc": datetime.now(timezone.utc).isoformat(),
    "status": "final_test_evaluated" if RUN_MODE == "final" else "validation_only_ablation_test_not_run",
    "variant": variant_name, "run_mode": RUN_MODE,
    "dataset": {
        "source": project_config["source"], "source_provenance_limitation": project_config["source"]["provenance_note"],
        "archive_sha256": archive_sha256, "verified_image_count": len(manifest),
        "verified_class_count": NUM_CLASSES, "partition_counts": {k: int(v) for k, v in partition_counts.items()},
        "class_distribution_by_split": class_distribution_by_split.to_dict(orient="records"),
        "split_manifest_path": str(SPLIT_CSV), "split_manifest_sha256": manifest_sha256,
        "duplicate_group_isolation": "verified; no group spans partitions",
    },
    "model": {
        "architecture": "ImageNet EfficientNetV2B0, optional ECA before GAP, BatchNormalization, Dense(256,relu), Dropout(0.5), Dense(38,softmax)",
        "attention": {"enabled": USE_ECA_ATTENTION, "type": "ECA", "placement": "backbone feature maps before global average pooling",
                      "channel_conv_kernel_size": 3},
        "preprocessing": preprocessing_config, "parameter_counts": parameter_summary,
        "input_shape": list(best_model.input_shape), "output_shape": list(best_model.output_shape),
    },
    "training_config": training_config, "training_history": training_history,
    "validation_results": validation_metrics, "test_results": test_metrics,
    "baseline_test_reference": baseline_test_metrics, "efficiency": efficiency,
    "limitations": [
        "PlantVillage images are controlled, mostly isolated leaves and may not represent field conditions, backgrounds, lighting, cameras, or disease severity.",
        "The local ZIP's exact Kaggle origin/version and license were not verified; confirm them before publication.",
        "Seeds and deterministic ops improve repeatability but hardware, TensorFlow/CUDA versions, and nondeterministic kernels can affect exact results.",
        "Ablation conclusions should use validation results; test metrics are for final reporting only.",
    ],
}
(RUN_OUTPUT_DIR / "enhanced_report.json").write_text(json.dumps(enhanced_report, indent=2), encoding="utf-8")
print("Saved enhanced report:", RUN_OUTPUT_DIR / "enhanced_report.json")
if RUN_MODE == "validation_ablation":
    print("Validation-only ablation complete. No test image dataset was created or evaluated.")

## Run artifacts

Each timestamped variant folder is under MyDrive/AgroVision/models/enhanced_efficientnetv2b0/. It contains the best validation checkpoint, optional HDF5 export, training config/history, class labels and split/config snapshots, preprocessing config, model summary, plots, validation metrics and confusion matrix, efficiency data, ablation comparison, and enhanced_report.json. Final mode additionally writes test metrics and baseline_vs_enhanced.json. Validation-only ablation reports use a null test_results field; no metric is invented.

No cell was run while preparing this notebook. Training begins only when you run it in Colab.